<div align="center">

<img src="https://raw.githubusercontent.com/winstonsmith1897/DantinoX/main/docs/images/dantinox.png" width="150" alt="DantinoX"/>

</div>

# DantinoX · 12 — Custom Paradigm Tutorial

<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/12_custom_paradigm.ipynb) &nbsp;
[![PyPI](https://img.shields.io/pypi/v/dantinox?color=7c3aed)](https://pypi.org/project/dantinox/) &nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-181717?logo=github)](https://github.com/winstonsmith1897/DantinoX)

</div>

*Extend DantinoX with your own generation algorithm — build a Semi-Autoregressive paradigm end to end.*

---

**You’ll learn**
- The `Paradigm` interface — DantinoX's extension point
- Implement a Semi-AR block-parallel decoder
- Train · generate · block-size ablation
- Register it into the paradigm system

**Runtime** — ~20 min · GPU (T4)

**What you'll build:** a **Semi-Autoregressive (SemiAR)** paradigm that generates `block_size` tokens in parallel per step, left-to-right:

```
Step 0:  [▒ ▒ ▒ ▒ · · ·]        predict first block
Step 1:  [w₁ w₂ w₃ w₄ ▒ ▒ ▒ ·]   next block
```

---

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import jax

print('Devices:', jax.devices())

  warnings.warn(


Devices: [CudaDevice(id=0)]


In [ ]:
!pip install -q uv
!uv pip install --system -q -U "dantinox[data,hub,elf,benchmark]" "flax>=0.12,<0.13" "jax[cuda12]"

In [2]:
import dantinox as dx

dx.doctor()   # environment health check — jax/flax/CUDA alignment, GPU visibility


  ████                █    █                █   █
  █   █  ███  ████  █████       ████   ███   █ █ 
  █   █ █   █ █   █   █    █    █   █ █   █   █  
  █   █ █  ██ █   █   █    █    █   █ █   █  █ █ 
  ████   ████ █   █   ██   ███  █   █  ███  █   █

  JAX/Flax transformer library  v0.4.7



DantinoX doctor
  dantinox             0.4.7
  jax                  0.9.2
  jaxlib               0.9.2
  flax                 0.12.6
  optax                0.2.8
  jax-cuda12-plugin    0.10.0
  jax-cuda12-pjrt      0.10.0
  transformers         4.44.2
  datasets             4.8.5
  devices              cuda:0
  ✗ jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"


{'versions': {'dantinox': '0.4.7',
  'jax': '0.9.2',
  'jaxlib': '0.9.2',
  'flax': '0.12.6',
  'optax': '0.2.8',
  'jax-cuda12-plugin': '0.10.0',
  'jax-cuda12-pjrt': '0.10.0',
  'transformers': '4.44.2',
  'datasets': '4.8.5'},
 'problems': ['jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"'],
 'warnings': [],
 'gpu': ['cuda:0'],
 'ok': False}

In [3]:
import os
import urllib.request

if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

tiny_shakespeare.txt already present


## 1 — The Paradigm interface

```python
class BaseParadigm(abc.ABC):
    def build_model(self, rngs: nnx.Rngs): ...      # required
    def loss(self, model, batch, rng): ...           # required
    def generate(self, model, *, max_new_tokens, **kw): ...  # required
    def stream(self, model, *, max_new_tokens, **kw): ...    # required
```

`Trainer` only calls `build_model()` and `loss()`.
`stream()` / `generate()` are inference-only.

## 2 — SemiAR design

**Training**: randomly mask `mask_rate` tokens, train to predict them (same as discrete diffusion).

**Generation**:
```
tokens = [MASK] * T
for block in range(T // block_size):
    s, e = block*block_size, (block+1)*block_size
    logits       = model(tokens)        # full bidirectional pass
    tokens[s:e]  = argmax(logits[s:e])  # decode this block only
```

Larger `block_size` → fewer steps → faster but less coherent.

In [4]:
from collections.abc import Iterator

import jax
import jax.numpy as jnp

import dantinox as dx

BASE_CFG = dx.ModelConfig(
    paradigm='discrete', dim=128, n_heads=4, num_blocks=4,
    causal=False, dropout=0.1,
)


class SemiARParadigm(dx.Paradigm):
    """Semi-autoregressive: decodes one block per step, left to right.

    Parameters
    ----------
    model_config : dx.ModelConfig   Must have causal=False.
    block_size   : int              Tokens decoded per step.
    mask_rate    : float            Fraction masked during training.
    """

    def __init__(self, model_config, block_size=4, mask_rate=0.5):
        super().__init__(model_config)
        self.block_size    = block_size
        self.mask_rate     = mask_rate
        self.mask_token_id = 1   # updated after tokenizer is loaded

    def build_model(self, rngs):
        from dantinox.core.model import Transformer
        return Transformer(self.model_config, rngs=rngs)

    def loss(self, model, batch, rng):
        B, T   = batch.shape
        rng, s = jax.random.split(rng)
        mask   = jax.random.uniform(s, (B, T)) < self.mask_rate
        x_t    = jnp.where(mask, self.mask_token_id, batch)
        logits = model(x_t, deterministic=False).logits
        lp     = jax.nn.log_softmax(logits, -1)
        nll    = -lp[jnp.arange(B)[:,None], jnp.arange(T)[None,:], batch]
        return (nll * mask).sum() / jnp.maximum(mask.sum(), 1.0)

    def stream(self, model, *, rng=None, max_new_tokens=64, **kwargs) -> Iterator:
        if rng is None: rng = jax.random.PRNGKey(0)
        n_blocks = max_new_tokens // self.block_size
        tokens   = jnp.full((1, max_new_tokens), self.mask_token_id, jnp.int32)
        for i in range(n_blocks):
            s, e    = i * self.block_size, (i+1) * self.block_size
            logits  = model(tokens, deterministic=True).logits
            new_tok = jnp.argmax(logits[0, s:e, :], axis=-1)
            tokens  = tokens.at[0, s:e].set(new_tok)
            yield i, n_blocks, tokens

    def generate(self, model, *, rng=None, max_new_tokens=64, **kwargs):
        result = None
        for _, _, tok in self.stream(model, rng=rng, max_new_tokens=max_new_tokens):
            result = tok
        return result


print('SemiARParadigm defined.')

SemiARParadigm defined.


## 3 — Training

`dx.Trainer` only needs `build_model()` and `loss()`.

In [5]:
semiar    = SemiARParadigm(BASE_CFG, block_size=4, mask_rate=0.5)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=2, batch_size=16, tokenizer_type='char')
run_dir   = dx.Trainer(semiar, train_cfg).fit('tiny_shakespeare.txt')
print('Checkpoint:', run_dir)


  ──────────────────────────────────────────────────────────────
  discrete  ·  bidirectional
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260715_114208
  parameters    1.1 M  (1,063,296)

  ── model ─────────────────────────────────────────────────────
  128-dim  ·  4h×32  ·  4 blocks  ·  vocab=66  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  bidirectional  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      2 epochs  ·  122 steps/epoch  ·  244 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 244 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/2:   0%|          | 0/122 [00:00<?, ?it/s]

...

Epoch 2/2:  93%|█████████▎| 114/122 [00:01<00:00, 94.13it/s, loss=3.3543]

Epoch 2/2:  93%|█████████▎| 114/122 [00:01<00:00, 94.13it/s, loss=3.3458]

  Epoch 2/2  train=3.2993  val=3.3387 (ppl 28.2)  ★ best  1.5s  676.8k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 3.3387
  saved → runs/20260715_114208
  ──────────────────────────────────────────────────────────────



Checkpoint: runs/20260715_114208


## 4 — Streaming generation

Each printed line shows the revealed tokens after one more block is decoded.

In [6]:
import os

from dantinox.utils.tokenizer import load_tokenizer_from_file

model = dx.load(run_dir, paradigm=semiar)
tok   = load_tokenizer_from_file(os.path.join(run_dir, 'tokenizer.json'))
if hasattr(tok, 'mask_token_id'):
    semiar.mask_token_id = tok.mask_token_id

MAX_NEW = 32
print(f'Generating {MAX_NEW} tokens  (block_size={semiar.block_size})\n')
for step, total, tokens in semiar.stream(model, max_new_tokens=MAX_NEW):
    revealed = tok.decode(tokens[0, :(step+1)*semiar.block_size].tolist())
    print(f'Block {step+1:2d}/{total} │ {revealed!r}')
print('\nFinal:', tok.decode(tokens[0].tolist()))

Generating 32 tokens  (block_size=4)



Block  1/8 │ '    '
Block  2/8 │ '        '


Block  3/8 │ '            '
Block  4/8 │ '                '


Block  5/8 │ '                    '
Block  6/8 │ '                        '


Block  7/8 │ '                            '
Block  8/8 │ '                                '

Final:                                 


## 5 — Block-size ablation

| `block_size` | Steps | Behaviour |
|---|---|---|
| `1` | T | One token at a time — iterative masked LM |
| `4–8` | T/4–T/8 | Sweet spot: fast + quality |
| `T` | 1 | One-shot — single-step discrete diffusion |

In [7]:
import time

import pandas as pd

MAX_TOKENS, rows = 64, []
for bsz in (1, 4, 8, 16, MAX_TOKENS):
    p = SemiARParadigm(BASE_CFG, block_size=bsz)
    m = dx.load(run_dir, paradigm=p)
    p.mask_token_id = semiar.mask_token_id
    t0 = time.perf_counter()
    for _ in range(5): p.generate(m, max_new_tokens=MAX_TOKENS)
    elapsed = (time.perf_counter()-t0)/5
    rows.append({'block_size':bsz,'n_steps':MAX_TOKENS//bsz,
                 'time_s':round(elapsed,4),'tok_s':round(MAX_TOKENS/elapsed,1)})
print(pd.DataFrame(rows).set_index('block_size').to_string())

            n_steps  time_s  tok_s
block_size                        
1                64  5.7483   11.1
4                16  1.1632   55.0
8                 8  0.6013  106.4
16                4  0.3409  187.7
64                1  0.1212  527.9


## 6 — Registering in the paradigm registry

Enables `dx.fit('semiar', ...)` and `dx.Paradigm(ModelConfig(paradigm='semiar'))`.

In [8]:
try:
    from dantinox.core.paradigm_registry import register_paradigm

    @register_paradigm('semiar')
    class SemiARRegistered(SemiARParadigm): pass

    print("Registered 'semiar'.")
    if hasattr(dx, 'list_paradigms'):
        print('Available paradigms:', dx.list_paradigms())

    # run = dx.fit('semiar', 'tiny_shakespeare.txt', dim=128, n_heads=4, num_blocks=4)
    # p   = dx.Paradigm(dx.ModelConfig(paradigm='semiar'))
    # m   = dx.load(run, paradigm=p)
except ImportError:
    print('registry API not available in this version — skipped.')

registry API not available in this version — skipped.


## 7 — Extension ideas

**Temperature sampling** instead of argmax:
```python
logits_b = logits[0, s:e, :] / temperature
new_tok  = jax.random.categorical(rng, jnp.log(jax.nn.softmax(logits_b, -1)))
```

**Confidence gating** — re-mask low-confidence tokens:
```python
conf    = jax.nn.softmax(logits[0, s:e], -1).max(-1)
new_tok = jnp.where(conf > threshold, argmax_tok, mask_id)
```

**Block-structured training** — always mask future blocks:
```python
start = jax.random.randint(rng, (), 0, T // block_size) * block_size
mask  = jnp.arange(T) >= start
```

**LoRA adapters** — works out of the box:
```python
lora_cfg = dataclasses.replace(BASE_CFG, use_lora=True, lora_rank=8, lora_targets='all')
semiar   = SemiARParadigm(lora_cfg, block_size=4)
```

**Swap backbone** for GQA or MoE:
```python
class SemiAR_GQA(SemiARParadigm):
    def build_model(self, rngs):
        from dantinox.core.model import Transformer
        cfg = dataclasses.replace(self.model_config, attention='gqa', kv_heads=2)
        return Transformer(cfg, rngs=rngs)
```

---

**Recap** — you learned:
- the `Paradigm` interface as DantinoX's extension point
- building, training, and registering a Semi-Autoregressive decoder

**Next →** [Documentation](https://dantinox.readthedocs.io) · [GitHub](https://github.com/winstonsmith1897/DantinoX) — issues and PRs welcome.